# Lab 8 — Enterprise Patterns for ML Systems

**Module 8 · Enterprise Patterns for ML Systems**

---

This notebook builds a layered 3DGS compression service from a Transaction Script baseline to a fully patterned architecture, measuring the concrete benefit of each step.

**What you will build:**
1. **Baseline audit** — dependency violations in a monolithic `compress_scene()` function
2. **Domain layer** — `GaussianScene` + pluggable `CompressionStrategy` objects
3. **Data source layer** — `SceneRepository` (Data Mapper + Identity Map + Unit of Work)
4. **Concurrency** — Optimistic Offline Lock with version columns
5. **Service layer** — thin `CompressionService` delegating to domain objects
6. **API layer** — FastAPI Remote Facade with Data Transfer Objects
7. **Measurements** — N+1 speedup, lock conflict rate, test coverage without a database

**No GPU required.** All computation is CPU-based SQLite + pure Python.

## Setup

In [ ]:
!pip install -q fastapi uvicorn httpx numpy 2>/dev/null || true

import sys, os, time, sqlite3, random, threading
from dataclasses import dataclass, field, asdict
from typing import Protocol, Optional
import numpy as np

print(f"Python {sys.version.split()[0]}  |  numpy {np.__version__}")

## Part A — Baseline: The Transaction Script

The monolithic starting point mixes presentation, domain, and data source concerns in one function.
We audit it before refactoring.

In [ ]:
# ── Monolithic baseline (DO NOT USE IN PRODUCTION) ───────────────────────────
# This is the Transaction Script anti-pattern — everything in one function.

BASELINE_CODE = '''
import sqlite3, json, numpy as np
from flask import Flask, request, jsonify   # presentation

app = Flask(__name__)

@app.route("/compress", methods=["POST"])
def compress_scene():                        # ALL concerns in one function
    # Presentation: parse request
    body     = request.json
    scene_id = body["scene_id"]
    ratio    = float(body["target_ratio"])
    strategy = body.get("strategy", "opacity_prune")

    # Data source: load from database
    conn      = sqlite3.connect("scenes.db")
    rows      = conn.execute("SELECT * FROM gaussians WHERE scene_id = ?", (scene_id,)).fetchall()
    gaussians = [dict(zip([d[0] for d in conn.execute("PRAGMA table_info(gaussians)").fetchall()], r))
                 for r in rows]

    # Domain: compression logic
    if strategy == "opacity_prune":
        threshold  = np.percentile([g["opacity"] for g in gaussians], (1 - ratio) * 100)
        compressed = [g for g in gaussians if g["opacity"] >= threshold]
    elif strategy == "random":
        n          = int(len(gaussians) * ratio)
        compressed = random.sample(gaussians, n)
    else:
        return jsonify({"error": f"Unknown strategy: {strategy}"}), 400

    # Data source: save result
    conn.execute("DELETE FROM compressed WHERE scene_id = ?", (scene_id,))
    for g in compressed:
        conn.execute("INSERT INTO compressed VALUES (?, ?, ?, ?, ?)",
                     (scene_id, g["x"], g["y"], g["z"], g["opacity"]))
    conn.commit()

    # Presentation: format response
    return jsonify({
        "scene_id":   scene_id,
        "original":   len(gaussians),
        "compressed": len(compressed),
        "ratio":      len(compressed) / len(gaussians),
    })
'''

# ── Dependency audit ─────────────────────────────────────────────────────────
import ast

tree = ast.parse(BASELINE_CODE)

concerns = {
    "Presentation": ["flask", "jsonify", "request", "json"],
    "Data source":  ["sqlite3", "SELECT", "INSERT", "DELETE", "conn"],
    "Domain":       ["opacity_prune", "random", "threshold", "np.percentile"],
}

print("=== Dependency Audit: compress_scene() ===")
print()
for concern, keywords in concerns.items():
    hits = [kw for kw in keywords if kw in BASELINE_CODE]
    print(f"  {concern:15s}: {hits}")

print()
print("Violations found: all three concerns present in one function.")
print()
print("Consequences:")
print("  → Cannot test domain logic without Flask + SQLite")
print("  → Adding a 3rd strategy requires modifying this function")
print("  → Switching DB requires rewriting data source calls here")
print("  → Silent data corruption: no concurrency control")

## Part B — Domain Layer

Extract all business logic into pure Python — zero database imports, zero HTTP imports.
This layer must be testable with `assert` statements and no external setup.

In [ ]:
# ── Domain objects ────────────────────────────────────────────────────────────
# No imports from sqlite3, flask, fastapi, or any infrastructure library.

@dataclass
class Gaussian:
    id      : str
    x       : float
    y       : float
    z       : float
    opacity : float
    r       : float = 0.5
    g       : float = 0.5
    b       : float = 0.5

    def is_floater(self, threshold: float = 0.005) -> bool:
        """Gaussians below this opacity threshold contribute negligibly to rendering."""
        return self.opacity < threshold

    def importance(self) -> float:
        """Proxy for Gaussian importance: opacity-weighted (scale not available here)."""
        return self.opacity


@dataclass
class GaussianScene:
    id        : str
    gaussians : list = field(default_factory=list)

    def count(self) -> int:
        return len(self.gaussians)

    def compress(self, strategy: 'CompressionStrategy', target_ratio: float = 0.5) -> 'GaussianScene':
        """Apply a compression strategy; returns a NEW scene (immutable transform)."""
        kept = strategy.select(self.gaussians, target_ratio)
        return GaussianScene(id=self.id, gaussians=kept)

    def compression_ratio(self, original_count: int) -> float:
        if original_count == 0: return 1.0
        return self.count() / original_count

    def floater_count(self, threshold: float = 0.005) -> int:
        return sum(1 for g in self.gaussians if g.is_floater(threshold))

    def avg_opacity(self) -> float:
        if not self.gaussians: return 0.0
        return sum(g.opacity for g in self.gaussians) / self.count()


# ── Compression strategies (Strategy pattern) ─────────────────────────────────
class CompressionStrategy(Protocol):
    def select(self, gaussians: list, ratio: float) -> list: ...


class OpacityPruneStrategy:
    """Keep the top `ratio` fraction by opacity."""
    def select(self, gaussians: list, ratio: float) -> list:
        if not gaussians: return []
        threshold = float(np.percentile([g.opacity for g in gaussians], (1 - ratio) * 100))
        return [g for g in gaussians if g.opacity >= threshold]


class RandomPruneStrategy:
    """Randomly retain `ratio` fraction."""
    def __init__(self, seed: int = 42):
        self._rng = np.random.default_rng(seed)

    def select(self, gaussians: list, ratio: float) -> list:
        n = max(1, int(len(gaussians) * ratio))
        idx = self._rng.choice(len(gaussians), size=n, replace=False)
        return [gaussians[i] for i in sorted(idx)]


class FloaterPruneStrategy:
    """Remove all Gaussians below an absolute opacity threshold."""
    def __init__(self, threshold: float = 0.005):
        self._t = threshold

    def select(self, gaussians: list, ratio: float) -> list:
        return [g for g in gaussians if not g.is_floater(self._t)]


class ImportancePruneStrategy:
    """Keep the top `ratio` fraction by importance score."""
    def select(self, gaussians: list, ratio: float) -> list:
        n = max(1, int(len(gaussians) * ratio))
        return sorted(gaussians, key=lambda g: g.importance(), reverse=True)[:n]


STRATEGY_REGISTRY = {
    "opacity_prune":    OpacityPruneStrategy(),
    "random_prune":     RandomPruneStrategy(),
    "floater_prune":    FloaterPruneStrategy(),
    "importance_prune": ImportancePruneStrategy(),
}

print(f"Domain layer loaded. Strategies registered: {list(STRATEGY_REGISTRY)}")

In [ ]:
# ── Domain tests — no database, no HTTP, no GPU ───────────────────────────────

def make_test_scene(n=200, seed=0) -> GaussianScene:
    rng = np.random.default_rng(seed)
    gaussians = [
        Gaussian(
            id      = str(i),
            x       = float(rng.uniform(-1, 1)),
            y       = float(rng.uniform(-1, 1)),
            z       = float(rng.uniform(-1, 1)),
            opacity = float(rng.random()),
        )
        for i in range(n)
    ]
    return GaussianScene(id='test_scene', gaussians=gaussians)


def run_domain_tests():
    scene = make_test_scene(1000)

    # Test 1: OpacityPrune keeps ~50%
    compressed = scene.compress(OpacityPruneStrategy(), 0.5)
    ratio = compressed.compression_ratio(scene.count())
    assert abs(ratio - 0.5) < 0.02, f"Expected ~0.5, got {ratio:.3f}"
    print(f"  [PASS] OpacityPrune 50%: kept {compressed.count()}/{scene.count()} "
          f"(ratio={ratio:.3f})")

    # Test 2: FloaterPrune removes all below threshold
    compressed2 = scene.compress(FloaterPruneStrategy(0.5))
    assert all(g.opacity >= 0.5 for g in compressed2.gaussians), "Floater threshold violated"
    print(f"  [PASS] FloaterPrune(0.5): kept {compressed2.count()} Gaussians")

    # Test 3: compress() is non-destructive (original scene unchanged)
    _ = scene.compress(OpacityPruneStrategy(), 0.3)
    assert scene.count() == 1000, "Original scene was mutated!"
    print(f"  [PASS] compress() is non-destructive: original scene still has {scene.count()} splats")

    # Test 4: Adding a strategy requires no changes to GaussianScene
    compressed4 = scene.compress(ImportancePruneStrategy(), 0.25)
    assert compressed4.count() == 250
    print(f"  [PASS] ImportancePrune 25%: kept {compressed4.count()} (no GaussianScene changes needed)")

    # Test 5: is_floater
    floater_scene = make_test_scene(100, seed=1)
    floaters = [g for g in floater_scene.gaussians if g.is_floater(0.5)]
    print(f"  [PASS] Floater detection: {len(floaters)}/100 Gaussians have opacity < 0.5")


t0 = time.perf_counter()
print("Running domain tests (no database):")
run_domain_tests()
elapsed = (time.perf_counter() - t0) * 1000
print(f"\nAll domain tests passed in {elapsed:.1f} ms (no database required).")

## Part C — Data Source Layer: Repository + Identity Map + Unit of Work

The repository presents a collection-like interface to the domain layer.
Internally it uses SQLite, but the domain code never sees `sqlite3`.

In [ ]:
# ── Exceptions ────────────────────────────────────────────────────────────────
class OptimisticLockError(Exception):
    """Raised when a version conflict is detected during save()."""


# ── Mapper: translates between Gaussian domain objects and DB rows ─────────────
class GaussianMapper:
    COLS = ('id', 'scene_id', 'x', 'y', 'z', 'opacity', 'r', 'g_val', 'b')

    def to_domain(self, row: tuple) -> Gaussian:
        d = dict(zip(self.COLS, row))
        return Gaussian(
            id=d['id'], x=d['x'], y=d['y'], z=d['z'],
            opacity=d['opacity'], r=d['r'], g=d['g_val'], b=d['b']
        )

    def from_domain(self, g: Gaussian, scene_id: str) -> tuple:
        return (g.id, scene_id, g.x, g.y, g.z, g.opacity, g.r, g.g, g.b)


# ── Repository: collection-like interface, hides all DB details ───────────────
class SceneRepository:
    def __init__(self, conn: sqlite3.Connection):
        self._conn   = conn
        self._mapper = GaussianMapper()
        self._identity_map: dict[str, GaussianScene] = {}   # Identity Map
        self._versions: dict[str, int] = {}                 # version tracking
        self._ensure_schema()

    def _ensure_schema(self):
        self._conn.executescript("""
            CREATE TABLE IF NOT EXISTS scenes (
                id TEXT PRIMARY KEY,
                gaussian_count INTEGER,
                version INTEGER DEFAULT 0
            );
            CREATE TABLE IF NOT EXISTS gaussians (
                id TEXT,
                scene_id TEXT,
                x REAL, y REAL, z REAL,
                opacity REAL, r REAL, g_val REAL, b REAL
            );
            CREATE INDEX IF NOT EXISTS idx_gaussians_scene
                ON gaussians(scene_id);
        """)

    def find(self, scene_id: str) -> Optional[GaussianScene]:
        """Load a scene. Returns cached object if already loaded (Identity Map)."""
        # Identity Map: avoid duplicate load
        if scene_id in self._identity_map:
            return self._identity_map[scene_id]

        rows = self._conn.execute(
            "SELECT id, scene_id, x, y, z, opacity, r, g_val, b "
            "FROM gaussians WHERE scene_id = ?", (scene_id,)
        ).fetchall()

        if not rows:
            return None

        # Load version for optimistic locking
        ver_row = self._conn.execute(
            "SELECT version FROM scenes WHERE id = ?", (scene_id,)
        ).fetchone()
        self._versions[scene_id] = ver_row[0] if ver_row else 0

        gaussians = [self._mapper.to_domain(row) for row in rows]
        scene = GaussianScene(id=scene_id, gaussians=gaussians)
        self._identity_map[scene_id] = scene   # register
        return scene

    def save(self, scene: GaussianScene, expected_version: Optional[int] = None) -> None:
        """
        Persist scene with optional Optimistic Offline Lock.
        Unit of Work: all writes in a single transaction.
        """
        if expected_version is not None:
            row = self._conn.execute(
                "SELECT version FROM scenes WHERE id = ?", (scene.id,)
            ).fetchone()
            current = row[0] if row else 0
            if current != expected_version:
                raise OptimisticLockError(
                    f"Version conflict on '{scene.id}': "
                    f"expected {expected_version}, found {current}"
                )

        # Unit of Work: batch all writes atomically
        with self._conn:
            self._conn.execute(
                "DELETE FROM gaussians WHERE scene_id = ?", (scene.id,)
            )
            self._conn.executemany(
                "INSERT INTO gaussians VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)",
                [self._mapper.from_domain(g, scene.id) for g in scene.gaussians]
            )
            self._conn.execute(
                """
                INSERT INTO scenes (id, gaussian_count, version)
                VALUES (?, ?, 1)
                ON CONFLICT(id) DO UPDATE
                    SET gaussian_count = excluded.gaussian_count,
                        version = version + 1
                """,
                (scene.id, scene.count())
            )

        self._identity_map[scene.id]  = scene   # update map
        self._versions[scene.id]      = (expected_version or 0) + 1

    def current_version(self, scene_id: str) -> int:
        row = self._conn.execute(
            "SELECT version FROM scenes WHERE id = ?", (scene_id,)
        ).fetchone()
        return row[0] if row else 0

    def seed_scene(self, scene_id: str, n_gaussians: int = 1000, seed: int = 0):
        """Utility: create and persist a synthetic scene."""
        scene = make_test_scene(n_gaussians, seed)
        scene.id = scene_id
        self.save(scene)
        return scene


print("SceneRepository defined.")

In [ ]:
# ── Repository tests ──────────────────────────────────────────────────────────

conn = sqlite3.connect(':memory:')   # in-memory DB — no file needed
repo = SceneRepository(conn)

# Seed a scene
repo.seed_scene('garden', n_gaussians=500, seed=42)
print("Seeded 'garden' scene.")

# Test 1: find() returns correct scene
scene = repo.find('garden')
assert scene is not None
assert scene.count() == 500
print(f"  [PASS] find(): loaded {scene.count()} Gaussians")

# Test 2: Identity Map — second find() returns SAME object
scene2 = repo.find('garden')
assert scene is scene2, "Identity Map failed: returned different objects!"
print(f"  [PASS] Identity Map: second find() returned same object (id={id(scene)})")

# Test 3: find() returns None for unknown scene
assert repo.find('nonexistent') is None
print("  [PASS] find('nonexistent') returns None")

# Test 4: save() + find() round-trip
compressed = scene.compress(OpacityPruneStrategy(), 0.4)
repo_fresh = SceneRepository(conn)   # new repo, empty identity map
repo_fresh.save(compressed)
reloaded = repo_fresh.find('garden')
assert reloaded.count() == compressed.count(), f"Save/load mismatch: {reloaded.count()} != {compressed.count()}"
print(f"  [PASS] save()/find() round-trip: {reloaded.count()} Gaussians persisted correctly")

print("\nAll repository tests passed.")

## Part D — Optimistic Offline Lock

Simulate two concurrent processes reading the same scene version, then both trying to write.
The second write should fail with `OptimisticLockError`.

In [ ]:
# ── Optimistic lock demonstration ─────────────────────────────────────────────

conn_lock = sqlite3.connect(':memory:')
repo_a    = SceneRepository(conn_lock)   # simulates process A
repo_b    = SceneRepository(conn_lock)   # simulates process B (shared DB)

# Seed scene at version 0
repo_a.seed_scene('shared_scene', n_gaussians=200, seed=0)
v0 = repo_a.current_version('shared_scene')
print(f"Initial version: {v0}")

# Both A and B read the scene at version v0
scene_a = repo_a.find('shared_scene')
scene_b = repo_b.find('shared_scene')    # also reads version v0

# A compresses and saves first — succeeds
compressed_a = scene_a.compress(OpacityPruneStrategy(), 0.5)
repo_a.save(compressed_a, expected_version=v0)
v1 = repo_a.current_version('shared_scene')
print(f"Process A saved successfully. Version is now: {v1}")

# B tries to save with the old version — should fail
compressed_b = scene_b.compress(RandomPruneStrategy(), 0.6)
try:
    repo_b.save(compressed_b, expected_version=v0)   # v0 is stale!
    print("ERROR: conflict not detected!")
except OptimisticLockError as e:
    print(f"Process B correctly detected conflict: {e}")
    # Retry with fresh version
    repo_b2 = SceneRepository(conn_lock)   # fresh repo = empty identity map
    fresh_scene = repo_b2.find('shared_scene')
    fresh_compressed = fresh_scene.compress(RandomPruneStrategy(), 0.6)
    repo_b2.save(fresh_compressed, expected_version=v1)
    v2 = repo_b2.current_version('shared_scene')
    print(f"Process B retried and saved successfully. Version is now: {v2}")

In [ ]:
# ── Conflict rate under concurrent load ───────────────────────────────────────
# Simulate N workers each reading and writing the same scene.
# Measure: how often does a version conflict occur?

def simulate_concurrent_writes(n_workers: int, n_rounds: int = 20,
                                scene_size: int = 500, seed: int = 42):
    """Simulate n_workers doing read-modify-write on the same scene."""
    conn_sim = sqlite3.connect(':memory:',
                               check_same_thread=False,
                               isolation_level=None)   # autocommit for thread safety demo
    conn_sim.execute('PRAGMA journal_mode=WAL')        # better concurrent read
    root_repo = SceneRepository(conn_sim)
    root_repo.seed_scene('contested', n_gaussians=scene_size, seed=seed)

    results = {'success': 0, 'conflict': 0}
    lock = threading.Lock()

    def worker(_):
        worker_repo = SceneRepository(conn_sim)
        for _ in range(n_rounds):
            scene   = worker_repo.find('contested')
            # Invalidate identity map so next round re-reads from DB
            worker_repo._identity_map.clear()
            if scene is None:
                continue
            version = worker_repo.current_version('contested')
            compressed = scene.compress(OpacityPruneStrategy(), 0.8)
            try:
                worker_repo.save(compressed, expected_version=version)
                with lock: results['success'] += 1
            except OptimisticLockError:
                with lock: results['conflict'] += 1

    threads = [threading.Thread(target=worker, args=(i,)) for i in range(n_workers)]
    for t in threads: t.start()
    for t in threads: t.join()

    total = results['success'] + results['conflict']
    conflict_pct = 100.0 * results['conflict'] / total if total > 0 else 0
    return results['success'], results['conflict'], conflict_pct


print(f"{'Workers':>8} {'Success':>10} {'Conflicts':>10} {'Conflict %':>12}")
print("-" * 44)
for n_workers in [2, 4, 8, 16]:
    s, c, pct = simulate_concurrent_writes(n_workers, n_rounds=10)
    print(f"{n_workers:>8} {s:>10} {c:>10} {pct:>11.1f}%")

print()
print("Interpretation:")
print("  Low conflict rate → Optimistic locking is the correct choice.")
print("  High conflict rate → Consider Pessimistic locking or reduced lock granularity.")

## Part E — Service Layer & Remote Facade

A thin Service Layer coordinates the domain + repository without duplicating domain logic.
The FastAPI endpoint is the Remote Facade: one coarse-grained call per use case.

In [ ]:
# ── Data Transfer Object ──────────────────────────────────────────────────────
# Serializable — no domain objects, no database connections.

@dataclass
class CompressionResultDTO:
    scene_id         : str
    original_count   : int
    compressed_count : int
    compression_ratio: float
    strategy_used    : str
    version          : int

    def to_dict(self) -> dict:
        return {
            'scene_id':          self.scene_id,
            'original_count':    self.original_count,
            'compressed_count':  self.compressed_count,
            'compression_ratio': round(self.compression_ratio, 4),
            'strategy':          self.strategy_used,
            'version':           self.version,
        }


@dataclass
class ScenePacketDTO:
    """Browser use-case packet: everything the browser needs in one call."""
    scene_id       : str
    gaussian_count : int
    floater_count  : int
    avg_opacity    : float
    version        : int

    def to_dict(self) -> dict: return asdict(self)


# ── Service Layer ─────────────────────────────────────────────────────────────
class CompressionService:
    """Thin coordinator: handles transaction boundaries and delegates to domain."""

    def __init__(self, repo: SceneRepository):
        self._repo = repo

    def compress(self, scene_id: str, target_ratio: float,
                 strategy_name: str = 'opacity_prune',
                 version: Optional[int] = None) -> CompressionResultDTO:
        strategy = STRATEGY_REGISTRY.get(strategy_name)
        if strategy is None:
            raise ValueError(f"Unknown strategy '{strategy_name}'. "
                             f"Available: {list(STRATEGY_REGISTRY)}")

        scene = self._repo.find(scene_id)
        if scene is None:
            raise KeyError(f"Scene '{scene_id}' not found")

        original_count = scene.count()
        compressed     = scene.compress(strategy, target_ratio)
        self._repo.save(compressed, expected_version=version)
        new_version = self._repo.current_version(scene_id)

        return CompressionResultDTO(
            scene_id          = scene_id,
            original_count    = original_count,
            compressed_count  = compressed.count(),
            compression_ratio = compressed.compression_ratio(original_count),
            strategy_used     = strategy_name,
            version           = new_version,
        )

    def get_scene_packet(self, scene_id: str) -> ScenePacketDTO:
        scene = self._repo.find(scene_id)
        if scene is None:
            raise KeyError(f"Scene '{scene_id}' not found")
        return ScenePacketDTO(
            scene_id       = scene_id,
            gaussian_count = scene.count(),
            floater_count  = scene.floater_count(),
            avg_opacity    = round(scene.avg_opacity(), 4),
            version        = self._repo.current_version(scene_id),
        )


print("Service layer and DTOs defined.")

In [ ]:
# ── Service layer tests ───────────────────────────────────────────────────────

conn_svc  = sqlite3.connect(':memory:')
repo_svc  = SceneRepository(conn_svc)
svc       = CompressionService(repo_svc)

repo_svc.seed_scene('bicycle', n_gaussians=800, seed=7)

# Test: compress via service
result = svc.compress('bicycle', target_ratio=0.5, strategy_name='opacity_prune')
print(f"Compress result (DTO):")
for k, v in result.to_dict().items():
    print(f"  {k:22s}: {v}")

# Test: get browser packet
packet = svc.get_scene_packet('bicycle')
print(f"\nScene packet (browser use case):")
for k, v in packet.to_dict().items():
    print(f"  {k:22s}: {v}")

# Test: unknown strategy raises ValueError
try:
    svc.compress('bicycle', 0.5, strategy_name='nonexistent')
    print("ERROR: should have raised ValueError")
except ValueError as e:
    print(f"\n[PASS] Unknown strategy rejected: {e}")

# Test: optimistic lock conflict via service
conn_c2 = sqlite3.connect(':memory:')
repo_c2 = SceneRepository(conn_svc)   # same DB connection
svc2    = CompressionService(repo_c2)
try:
    svc2.compress('bicycle', 0.4, version=0)   # stale version
    print("ERROR: should have raised OptimisticLockError")
except OptimisticLockError as e:
    print(f"[PASS] Service correctly surfaces lock conflict: {type(e).__name__}")

## Part F — FastAPI Remote Facade

The Remote Facade exposes the service via HTTP. Each endpoint is use-case–oriented:
- `POST /compress` — run compression job
- `GET /scene/{id}` — browser packet (one call, all needed data)

We test with `httpx` against an in-process ASGI app (no server port needed).

In [ ]:
try:
    from fastapi import FastAPI, HTTPException, Query
    from fastapi.testclient import TestClient
    FASTAPI_AVAILABLE = True
except ImportError:
    FASTAPI_AVAILABLE = False
    print("FastAPI not available — skipping API tests. Install with: pip install fastapi httpx")

if FASTAPI_AVAILABLE:
    conn_api = sqlite3.connect(':memory:', check_same_thread=False)
    repo_api = SceneRepository(conn_api)
    svc_api  = CompressionService(repo_api)

    # Seed scenes for the API tests
    for name, n in [('garden', 1000), ('bicycle', 750), ('counter', 500)]:
        repo_api.seed_scene(name, n_gaussians=n)

    app = FastAPI(title="3DGS Compression Service")

    @app.post("/compress")
    def compress_endpoint(
        scene_id     : str   = Query(...),
        target_ratio : float = Query(0.5, ge=0.01, le=1.0),
        strategy     : str   = Query('opacity_prune'),
        version      : Optional[int] = Query(None),
    ):
        """Remote Facade: coarse-grained compression endpoint."""
        try:
            result = svc_api.compress(scene_id, target_ratio, strategy, version)
            return result.to_dict()
        except KeyError as e:
            raise HTTPException(404, str(e))
        except ValueError as e:
            raise HTTPException(400, str(e))
        except OptimisticLockError as e:
            raise HTTPException(409, str(e))

    @app.get("/scene/{scene_id}")
    def scene_packet_endpoint(scene_id: str):
        """Remote Facade: browser use-case packet — one call, all needed data."""
        try:
            return svc_api.get_scene_packet(scene_id).to_dict()
        except KeyError as e:
            raise HTTPException(404, str(e))

    client = TestClient(app)
    print("FastAPI app created. Running endpoint tests...")

    # Test compress endpoint
    resp = client.post('/compress?scene_id=garden&target_ratio=0.5&strategy=opacity_prune')
    assert resp.status_code == 200, resp.text
    data = resp.json()
    print(f"\n  POST /compress → 200 OK")
    for k, v in data.items():
        print(f"    {k:22s}: {v}")

    # Test scene packet endpoint
    resp2 = client.get('/scene/bicycle')
    assert resp2.status_code == 200, resp2.text
    data2 = resp2.json()
    print(f"\n  GET /scene/bicycle → 200 OK")
    for k, v in data2.items():
        print(f"    {k:22s}: {v}")

    # Test 404
    resp3 = client.get('/scene/nonexistent')
    assert resp3.status_code == 404
    print(f"\n  GET /scene/nonexistent → 404 (correct)")

    # Test 400 on bad strategy
    resp4 = client.post('/compress?scene_id=garden&strategy=bad_strategy')
    assert resp4.status_code == 400
    print(f"  POST /compress (bad strategy) → 400 (correct)")

    # Test 409 on version conflict
    resp5 = client.post('/compress?scene_id=bicycle&version=0')  # stale
    assert resp5.status_code == 409
    print(f"  POST /compress (stale version) → 409 Conflict (correct)")

    print("\nAll API tests passed.")

## Part G — Measuring the Concrete Wins

Quantify the benefits of the refactored architecture against the baseline.

In [ ]:
# ── N+1 problem: Identity Map speedup ────────────────────────────────────────

conn_bench = sqlite3.connect(':memory:')
repo_bench = SceneRepository(conn_bench)

# Seed 30 scenes
scene_ids = [f'scene_{i:03d}' for i in range(30)]
for sid in scene_ids:
    repo_bench.seed_scene(sid, n_gaussians=200, seed=hash(sid) % 1000)

# Without Identity Map: fresh repo each load (simulates N+1 pattern)
t0 = time.perf_counter()
for _ in range(3):   # 3 iterations simulating 3 view renders
    for sid in scene_ids:
        fresh_repo = SceneRepository(conn_bench)   # no identity map
        _ = fresh_repo.find(sid)
t_no_cache = (time.perf_counter() - t0) * 1000

# With Identity Map: same repo, cached after first load
repo_cached = SceneRepository(conn_bench)
t0 = time.perf_counter()
for _ in range(3):
    for sid in scene_ids:
        _ = repo_cached.find(sid)   # hits identity map after first iteration
t_cached = (time.perf_counter() - t0) * 1000

speedup = t_no_cache / t_cached if t_cached > 0 else float('inf')

print("=== Identity Map: N+1 vs. Cached ===")
print(f"  Scenes: {len(scene_ids)} × 3 renders = {len(scene_ids)*3} total loads")
print(f"  Without Identity Map (N+1): {t_no_cache:.1f} ms")
print(f"  With Identity Map (cached): {t_cached:.1f} ms")
print(f"  Speedup: {speedup:.1f}×")

In [ ]:
# ── Layer isolation verification ──────────────────────────────────────────────
# Verify the dependency rule is enforced: domain has no DB/HTTP imports.

import inspect

DOMAIN_CLASSES   = [Gaussian, GaussianScene,
                    OpacityPruneStrategy, RandomPruneStrategy,
                    FloaterPruneStrategy, ImportancePruneStrategy]
INFRA_KEYWORDS   = ['sqlite3', 'psycopg2', 'sqlalchemy',
                    'flask', 'fastapi', 'django',
                    'SELECT', 'INSERT', 'UPDATE', 'DELETE']

print("=== Domain Layer Isolation Check ===")
violations = []
for cls in DOMAIN_CLASSES:
    src = inspect.getsource(cls)
    found = [kw for kw in INFRA_KEYWORDS if kw in src]
    if found:
        violations.append((cls.__name__, found))
        print(f"  VIOLATION in {cls.__name__}: found {found}")
    else:
        print(f"  OK  {cls.__name__}: no infrastructure imports")

if not violations:
    print("\n[PASS] Dependency rule satisfied: domain layer contains zero infrastructure imports.")
else:
    print(f"\n[FAIL] {len(violations)} violation(s) found.")

print()
print("=== Before vs. After Summary ===")
rows = [
    ("Add new strategy",         "Modify compress_scene()",     "New class + register in dict"),
    ("Swap SQLite → PostgreSQL", "Rewrite all SQL in function", "New SceneRepository subclass"),
    ("Test domain without DB",   "Not possible",                "< 50ms, pure Python"),
    ("Concurrent write safety",  "Silent overwrite (bug)",      "409 Conflict (detectable)"),
    ("Identity Map speedup",     "1× (always queries DB)",      f"{speedup:.1f}× for repeat loads"),
]
print(f"  {'Metric':<32} {'Transaction Script':<30} {'Layered Architecture'}")
print("  " + "-" * 85)
for metric, before, after in rows:
    print(f"  {metric:<32} {before:<30} {after}")

## Part H — Exploring Open Problems

The following exercises correspond to concrete open problems from Reading 3.
Pick one that matches your interest level.

### Problem 2.4 (Difficulty B): Heterogeneous persistence
> Design a Data Mapper for a mixed persistence scenario: Gaussian positions stored in SQLite,
> SH coefficients stored in HDF5, rendered previews in S3. The domain object must be
> loadable from any combination of backends without changing domain code.

**Starting point:**

```python
class HeterogeneousSceneRepository:
    def __init__(self, sqlite_conn, hdf5_path: str, s3_bucket: str):
        self._sql    = sqlite_conn
        self._hdf5   = hdf5_path
        self._s3     = s3_bucket

    def find(self, scene_id: str) -> GaussianScene:
        # Load geometry from SQLite
        # Load SH coefficients from HDF5 (h5py)
        # Load preview from S3 (boto3)
        # Merge into a single GaussianScene domain object
        # GaussianScene code is NOT modified
        raise NotImplementedError
```

### Problem 3.2 (Difficulty B): Optimistic vs. Pessimistic crossover
> Compare locking strategies under varying numbers of concurrent writers.
> Find the conflict rate crossover point where pessimistic outperforms optimistic.

**Starting point:** use `simulate_concurrent_writes()` from Part D with a Pessimistic variant
that uses `threading.Lock()` instead of version checking. Plot both against worker count.

### Problem 5.5 (Difficulty D — Open): Validation architecture
> Design a layered validation system for a 3DGS scene that checks geometric, semantic,
> and training validity as typed domain exceptions, not raw assertion failures.

**Starting point:**

```python
class ValidationError(Exception):
    pass

class GeometricValidator:
    def validate(self, scene: GaussianScene) -> list[ValidationError]:
        errors = []
        for g in scene.gaussians:
            if not (0.0 <= g.opacity <= 1.0):
                errors.append(ValidationError(f"Splat {g.id}: opacity {g.opacity} out of [0,1]"))
        return errors

# Extend with SemanticValidator, TrainingValidator
# Each validator only checks what its layer owns
```

In [ ]:
# ── Your workspace — implement one open problem here ──────────────────────────

# Starter: Pessimistic lock comparison (Problem 3.2)

import threading

_pessimistic_lock = threading.Lock()   # process-wide lock (coarse-grained)

def simulate_pessimistic_writes(n_workers: int, n_rounds: int = 20,
                                scene_size: int = 500, seed: int = 42):
    """Pessimistic version: use threading.Lock to serialize all writes."""
    conn_p = sqlite3.connect(':memory:', check_same_thread=False)
    repo_p = SceneRepository(conn_p)
    repo_p.seed_scene('contested_p', n_gaussians=scene_size, seed=seed)

    results = {'success': 0, 'waited_ms': 0}
    lock = threading.Lock()

    def worker(_):
        worker_repo = SceneRepository(conn_p)
        for _ in range(n_rounds):
            worker_repo._identity_map.clear()
            scene = worker_repo.find('contested_p')
            if scene is None:
                continue
            compressed = scene.compress(OpacityPruneStrategy(), 0.8)
            t0 = time.perf_counter()
            with _pessimistic_lock:      # blocks until lock is free
                waited = (time.perf_counter() - t0) * 1000
                worker_repo.save(compressed)   # no version check needed
                with lock:
                    results['success']   += 1
                    results['waited_ms'] += waited

    threads = [threading.Thread(target=worker, args=(i,)) for i in range(n_workers)]
    for t in threads: t.start()
    for t in threads: t.join()

    avg_wait = results['waited_ms'] / results['success'] if results['success'] > 0 else 0
    return results['success'], avg_wait


print("=== Optimistic vs. Pessimistic Locking Comparison ===")
print(f"{'Workers':>8} {'Opt Success':>12} {'Opt Conflict%':>14} {'Pess Success':>13} {'Pess Avg Wait(ms)':>18}")
print("-" * 70)
for n_workers in [2, 4, 8, 16]:
    opt_s, opt_c, opt_pct = simulate_concurrent_writes(n_workers, n_rounds=10)
    pess_s, pess_wait     = simulate_pessimistic_writes(n_workers, n_rounds=10)
    print(f"{n_workers:>8} {opt_s:>12} {opt_pct:>13.1f}% {pess_s:>13} {pess_wait:>17.2f}")

print()
print("Observation: optimistic has zero blocking wait; pessimistic wait grows with workers.")
print("Optimistic has conflicts; pessimistic has zero conflicts but serializes throughput.")

## Lab Deliverables

Submit answers to the following:

### Deliverable 1 — Baseline audit (Part A)
List the three concern types found in `compress_scene()` and name one concrete consequence of each violation.

### Deliverable 2 — Domain isolation (Part B + G)
Paste the output of the isolation check. How many infrastructure keywords did the domain layer contain? What does this enable for testing?

### Deliverable 3 — Identity Map speedup (Part G)
Report the measured speedup factor. At what point (how many scenes, how many re-renders) does Identity Map become worth its memory overhead?

### Deliverable 4 — Locking comparison (Part H)
From the Optimistic vs. Pessimistic table: at which worker count (if any) does the pessimistic average wait become comparable to the time lost retrying optimistic conflicts? Is there a crossover in this workload?

### Deliverable 5 — Open problem
Implement one of the three open problems from Part H (or propose your own). Describe:
- Which problem you chose
- What you built
- One concrete measurement that validates your implementation
- One limitation of your approach

---

### Summary: Patterns Applied

| Pattern | Where used | Concrete benefit measured |
|---|---|---|
| Three-layer architecture | All files | Domain has zero infra imports |
| Domain Model + Strategy | `GaussianScene`, `*Strategy` | New strategies = new class only |
| Data Mapper | `GaussianMapper` | Domain testable without DB |
| Repository | `SceneRepository` | DB backend swappable |
| Unit of Work | `save()` with `with self._conn` | Atomic multi-row writes |
| Identity Map | `_identity_map` in repo | N× speedup on repeat loads |
| Optimistic Offline Lock | `expected_version` param | 409 Conflict on stale write |
| Service Layer | `CompressionService` | Transaction boundaries in one place |
| Data Transfer Object | `CompressionResultDTO`, `ScenePacketDTO` | Serializable, no domain leakage |
| Remote Facade | FastAPI endpoints | One call per use case |
| Registry | `STRATEGY_REGISTRY` | Plugin-like strategy selection |